In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# dbutils.widgets.text('catalog', 'de_dev')

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.gold.product_kpi AS

WITH product_sales AS (

SELECT

p.product_id,

p.product_name,

p.category,

sp.supplier_name,

s.sale_date,

s.quantity,

s.sale_amount,

p.price

FROM ${catalog}.silver.sales_scd_1 s

JOIN ${catalog}.silver.products_scd_2 p

ON s.product_id=p.product_id

JOIN ${catalog}.silver.suppliers_scd_1 sp

ON p.supplier_id=sp.supplier_id

)

SELECT

product_id,

product_name,

category,

supplier_name,

sale_date,

quantity,

sale_amount,

price,

SUM(quantity)

OVER(

PARTITION BY category,product_id

)

AS total_quantity_sold,

SUM(sale_amount)

OVER(

PARTITION BY supplier_name

)

AS supplier_revenue,

AVG(sale_amount)

OVER(

PARTITION BY category

)

AS avg_category_sales,

PERCENT_RANK()

OVER(

ORDER BY SUM(sale_amount)

OVER(PARTITION BY product_id)

)

AS product_percent_rank

FROM product_sales;